# Krok 9. Czyszczenie danych i zimny start (E6)
**Wynik z instrukcji:** lista sytuacji psujących prognozę (nowe SKU, wycofane SKU, jednorazowe duże zakupy, korekty) i dla każdej decyzja, co z nią robimy. Poziom: SKU. Dla wszystkich wariantów.

**Zasada: oznaczamy, nie usuwamy.** Każda sytuacja dostaje **flagę**, a decyzja mówi, jak prognoza ma się z nią obchodzić. Dzięki temu nic nie znika z danych i każdy kolejny krok może zmienić decyzję bez ponownego czyszczenia.

**Moment odniesienia.** Wszystkie flagi liczymy „na dzień 31.12.2024” (koniec lat EDA), czyli tak, jak wyglądałyby w chwili prognozy na 2025.

| Sytuacja | Flaga | Jak wykrywamy | Decyzja (propozycja) |
|---|---|---|---|
| Nowe SKU (zimny start) | `flaga_nowe` | mniej niż 13 tygodni od pierwszej sprzedaży | bez własnej prognozy sezonowej: poziom z dostępnych tygodni, mnożniki asortymentu |
| Wycofane SKU | `flaga_wycofane` | brak sprzedaży od dłużej niż max(8 tygodni, 3 × ADI) | prognoza 0 |
| Jednorazowe duże zakupy | `flaga_jednorazowy` | dzień powyżej mediany + max(5 × 1,48 × MAD, 3 × mediana) przy co najmniej 20 dniach sprzedaży | do liczenia Bazy ucinamy do progu |
| Korekty remanentu | `flaga_korekta` | niezerowa `KorektaRemanentu` (jeśli kolumna jest w pliku) | nie są sprzedażą, zostawiamy poza prognozą; flaga do analizy braków |
| Towary przecenione (`AsId` 319) | brak (wyłączone) | `AsId == 319` | pomijamy w EDA, zgodnie ze specyfikacją |

**Wynik zapisany do:** `dane/eda/wynik_09_decyzje.csv`, `dane/eda/wynik_09_flagi_sku.csv` i `dane/eda/wynik_09_jednorazowe.csv`.

In [6]:
# Ustawienia wspólne: moduł eda_wspolne.py leży obok tego notebooka
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_wspolne as ew

ew.wejdz_do_korzenia()      # przechodzi do katalogu głównego projektu (tam, gdzie jest plik z danymi)
ew.styl()
print("Katalog projektu:", os.getcwd(), "| EDA używa danych do", ew.KONIEC_EDA.date(), "(rok testowy", ew.ROK_TEST, "pomijamy)")

Katalog projektu: /Users/mixer/python/analiza_sprzedazy_w_sklepie | EDA używa danych do 2024-12-31 (rok testowy 2025 pomijamy)


## Dane i wykluczony asortyment 319
#### Objaśnienie
**Po co.** Zaczynamy od sprzedaży (towar, dzień) i sprawdzamy, jak dużo wolumenu zdejmuje wyłączenie `AsId 319`. Wolumen asortymentu 319 liczymy z danych surowych (przed wyłączeniem), żeby wiedzieć, ile sprzedaży pomijamy.
**Jak czytać wynik.** Jeśli 319 to kilka procent wolumenu, pominięcie jest bezpieczne dla sklepu jako całości, ale w prognozie całego sklepu trzeba o nim pamiętać (dodać osobno).

In [7]:
import warnings
surowe = pd.read_parquet(ew.PLIK_REL, columns=["AsId", "Data", "DokId", "TypDok", ew.MIARA])
surowe = surowe[(surowe["DokId"] != -1) & (surowe["TypDok"] == 21) & (surowe["Data"] <= ew.KONIEC_EDA)]
udzial_319 = 100 * surowe.loc[surowe["AsId"].isin(ew.WYKLUCZ_AS), ew.MIARA].sum() / surowe[ew.MIARA].sum()
del surowe

fa = ew.wczytaj_sprzedaz()
kal = ew.kalendarz()
sd = fa.groupby(["SKU", "Data"], observed=True)[ew.MIARA].sum().reset_index(name="q")
tyg = ew.panel_tygodniowy(fa, kal, "SKU")
del fa
sd = sd[sd["q"] > 0]
wolumen = sd["q"].sum()
typ = ew.typ_popytu(tyg)
print(f"Asortyment {', '.join(map(str, ew.WYKLUCZ_AS))} wyłączony: {udzial_319:.1f} % wolumenu sprzedaży do końca 2024")
print(f"SKU-dni ze sprzedażą: {len(sd):,} | towarów: {sd['SKU'].nunique():,}".replace(",", " "))

Asortyment 319 wyłączony: 0.2 % wolumenu sprzedaży do końca 2024
SKU-dni ze sprzedażą: 1 061 489 | towarów: 9 954


## Nowe i wycofane SKU
#### Objaśnienie
**Po co.** Nowy towar nie ma historii, na której można oprzeć sezonowość i typ popytu (zimny start). Wycofany towar ma historię, ale nie ma już sprzedaży: prognoza z historii byłaby fałszywa.
**Co robi kod.**
- `tyg_od_pierwszej` i `tyg_od_ostatniej`: tygodnie od pierwszej i ostatniej sprzedaży do końca EDA.
- **Nowe**: `tyg_od_pierwszej < 13` (to samo 13 co minimum do klasyfikacji typu).
- **Wycofane**: brak sprzedaży dłuższy niż `max(8 tygodni, 3 × ADI)`. ADI to typowy odstęp między sprzedażami towaru, więc towar rzadko sprzedawany nie jest uznany za wycofany po krótkiej przerwie (próg dopasowuje się do towaru).
**Jak czytać wynik.** Dla każdej flagi mamy liczbę towarów i udział w wolumenie. Duża liczba, ale mały udział znaczy, że są to głównie towary marginalne. Duży udział wycofanych w wolumenie z całego okresu oznacza, że historia sporo zawiera sprzedaży towarów, których już nie będzie.

In [8]:
g = sd.groupby("SKU")["Data"].agg(["min", "max"])
flagi = pd.DataFrame({"pierwsza": g["min"], "ostatnia": g["max"]})
flagi["tyg_od_pierwszej"] = (ew.KONIEC_EDA - flagi["pierwsza"]).dt.days / 7
flagi["tyg_od_ostatniej"] = (ew.KONIEC_EDA - flagi["ostatnia"]).dt.days / 7
flagi["ADI"] = typ["ADI"].reindex(flagi.index)
flagi["sprzedaz"] = sd.groupby("SKU")["q"].sum()
flagi["flaga_nowe"] = flagi["tyg_od_pierwszej"] < ew.MIN_TYG_TYP
prog_wyc = np.maximum(8, 3 * flagi["ADI"].fillna(8))
flagi["flaga_wycofane"] = flagi["tyg_od_ostatniej"] > prog_wyc
def podsum(nazwa, maska):
    return {"sytuacja": nazwa, "towary": int(maska.sum()), "udzial_wolumenu_%": 100 * flagi.loc[maska, "sprzedaz"].sum() / wolumen}
print(pd.DataFrame([podsum("nowe SKU (< 13 tyg.)", flagi["flaga_nowe"]), podsum("wycofane SKU", flagi["flaga_wycofane"])]).round(2).to_string(index=False))

            sytuacja  towary  udzial_wolumenu_%
nowe SKU (< 13 tyg.)     526               0.30
        wycofane SKU    2623               8.99


## Jednorazowe duże zakupy
#### Objaśnienie
**Po co.** Jednorazowy duży zakup (np. hurtowy, na imprezę) nie powtórzy się, a podnosi średnią sprzedaży towaru i zawyża Bazę.
**Co robi kod.** Dla każdego towaru bierzemy dni ze sprzedażą i liczymy ich medianę `med` oraz **MAD** (medianę bezwzględnych odchyleń od mediany, odporną miarę rozrzutu). Dzień jest jednorazowy, gdy sprzedaż przekracza `med + max(5 × 1,4826 × MAD, 3 × med)` i towar ma co najmniej 20 takich dni (inaczej nie ma z czego liczyć mediany). Współczynnik 1,4826 przelicza MAD na odchylenie standardowe dla rozkładu normalnego. Dodatkowa składowa `3 × med` chroni towary o bardzo stałej sprzedaży (MAD bliski 0), w których każdy skok przekraczałby próg.
**Co z flagą.** Nie usuwamy dni. Zapisujemy próg: przy liczeniu Bazy sprzedaż dnia ucina się do tego progu.
**Jak czytać wynik.** Powinno to być bardzo mało dni (ułamek procenta), ale o zauważalnym wolumenie. Jeśli flaga łapie wiele zwykłych dni, próg jest zbyt niski.

In [9]:
sd["n_dni"] = sd.groupby("SKU")["q"].transform("size")
sd["med"] = sd.groupby("SKU")["q"].transform("median")
sd["dev"] = (sd["q"] - sd["med"]).abs()
sd["mad"] = sd.groupby("SKU")["dev"].transform("median")
sd["prog"] = sd["med"] + np.maximum(5 * 1.4826 * sd["mad"], 3 * sd["med"])
sd["flaga_jednorazowy"] = (sd["q"] > sd["prog"]) & (sd["n_dni"] >= 20)
jedn = sd[sd["flaga_jednorazowy"]][["SKU", "Data", "q", "prog"]].copy()
jedn["nadwyzka"] = jedn["q"] - jedn["prog"]
flagi["n_jednorazowych"] = jedn.groupby("SKU").size().reindex(flagi.index).fillna(0).astype(int)
flagi["flaga_jednorazowy"] = flagi["n_jednorazowych"] > 0
print(f"Dni oznaczone: {len(jedn):,} z {len(sd):,} ({100 * len(jedn) / len(sd):.2f} %) | towarów: {int(flagi['flaga_jednorazowy'].sum())} | "
      f"nadwyżka ponad próg: {100 * jedn['nadwyzka'].sum() / wolumen:.2f} % wolumenu".replace(",", " "))
print(jedn.sort_values("nadwyzka", ascending=False).head(8).round({"q": 1, "prog": 1, "nadwyzka": 1}).to_string(index=False))

Dni oznaczone: 10 318 z 1 061 489 (0.97 %) | towarów: 2694 | nadwyżka ponad próg: 1.24 % wolumenu
  SKU       Data     q  prog  nadwyzka
14957 2024-01-26 150.0  40.0     110.0
   75 2023-02-16 248.0 144.0     104.0
16866 2024-04-03 100.0   4.0      96.0
   75 2024-02-08 235.0 144.0      91.0
  752 2023-04-06  95.0  17.8      77.2
15114 2023-05-12 100.0  35.7      64.3
50008 2024-01-24  95.2  31.1      64.1
53351 2023-03-15  95.0  35.7      59.3


## Korekty remanentu i podsumowanie decyzji
#### Objaśnienie
**Po co.** Korekty remanentu to poprawki stanu magazynowego po inwentaryzacji. Nie są sprzedażą (filtr `TypDok == 21` ich nie zawiera), ale ich obecność mówi, które towary miały problemy ze stanem (np. braki, kradzieże, błędy przyjęć), a to bywa przyczyną „dziur” w sprzedaży. Kolumna `KorektaRemanentu` jest opcjonalna: jeśli jej nie ma w pliku, krok to pomija.
**Co robi kod.** Czyta z całego pliku (nie tylko ze sprzedaży) niezerowe korekty z lat EDA, liczy towary i dni, których dotyczą, i dopisuje flagę `flaga_korekta`. Na końcu składa tabelę decyzji i zapisuje wyniki.
**Jak czytać wynik.** Tabela decyzji to produkt końcowy kroku: dla każdej sytuacji liczba towarów lub dni, udział w wolumenie i uzgodniona reguła. Kolumna `udzial_wolumenu_%` pozwala ocenić, czy sytuacja warta jest osobnej logiki.

In [10]:
flagi["flaga_korekta"] = False
korekty_info = {"sytuacja": "korekty remanentu", "towary": 0, "udzial_wolumenu_%": np.nan}
if ew.ma_kolumne("KorektaRemanentu"):
    kr = pd.read_parquet(ew.PLIK_REL, columns=["SKU", "Data", "KorektaRemanentu"])
    kr = kr[(kr["Data"] <= ew.KONIEC_EDA) & (kr["KorektaRemanentu"].fillna(0) != 0)]
    sku_kor = set(kr["SKU"].unique())
    flagi["flaga_korekta"] = flagi.index.isin(sku_kor)
    korekty_info.update({"towary": int(flagi["flaga_korekta"].sum()),
                         "udzial_wolumenu_%": 100 * flagi.loc[flagi["flaga_korekta"], "sprzedaz"].sum() / wolumen})
    print(f"Korekty remanentu: {len(kr)} zapisów | towarów: {len(sku_kor)}")
else:
    print("Brak kolumny KorektaRemanentu w pliku: korekty pominięto.")

decyzje = pd.DataFrame([
    {**podsum("nowe SKU (zimny start)", flagi["flaga_nowe"]), "flaga": "flaga_nowe",
     "decyzja": "bez sezonowości własnej: poziom z dostępnych tygodni i mnożniki asortymentu"},
    {**podsum("wycofane SKU", flagi["flaga_wycofane"]), "flaga": "flaga_wycofane", "decyzja": "prognoza 0"},
    {"sytuacja": "jednorazowe duże zakupy", "towary": int(flagi["flaga_jednorazowy"].sum()),
     "udzial_wolumenu_%": 100 * jedn["nadwyzka"].sum() / wolumen, "flaga": "flaga_jednorazowy",
     "decyzja": "przy liczeniu Bazy ucinać do progu (zapisanego w pliku)"},
    {**korekty_info, "flaga": "flaga_korekta", "decyzja": "nie są sprzedażą; flaga do analizy braków magazynowych"},
    {"sytuacja": "towary przecenione (AsId " + ", ".join(map(str, ew.WYKLUCZ_AS)) + ")", "towary": np.nan, "udzial_wolumenu_%": udzial_319,
     "flaga": "wyłączone", "decyzja": "pomijamy w EDA (specyfikacja)"},
])
print(decyzje.round(2).to_string(index=False))

kol_f = ["pierwsza", "ostatnia", "tyg_od_pierwszej", "tyg_od_ostatniej", "ADI", "sprzedaz", "flaga_nowe", "flaga_wycofane",
         "n_jednorazowych", "flaga_jednorazowy", "flaga_korekta"]
ew.zapisz_wynik(decyzje, "wynik_09_decyzje.csv", index=False)
ew.zapisz_wynik(flagi[kol_f], "wynik_09_flagi_sku.csv")
ew.zapisz_wynik(jedn, "wynik_09_jednorazowe.csv", index=False)

Korekty remanentu: 28316 zapisów | towarów: 6878
                     sytuacja  towary  udzial_wolumenu_%             flaga                                                                     decyzja
       nowe SKU (zimny start)   526.0               0.30        flaga_nowe bez sezonowości własnej: poziom z dostępnych tygodni i mnożniki asortymentu
                 wycofane SKU  2623.0               8.99    flaga_wycofane                                                                  prognoza 0
      jednorazowe duże zakupy  2694.0               1.24 flaga_jednorazowy                     przy liczeniu Bazy ucinać do progu (zapisanego w pliku)
            korekty remanentu  6733.0              92.94     flaga_korekta                      nie są sprzedażą; flaga do analizy braków magazynowych
towary przecenione (AsId 319)     NaN               0.17         wyłączone                                               pomijamy w EDA (specyfikacja)
Zapisano: dane/eda/wynik_09_decyzje.csv (5 wi

'dane/eda/wynik_09_jednorazowe.csv'